In [1]:
import wrds
import time
import os
import pandas as pd

from src.path import OptionPath

In [2]:
from dotenv import load_dotenv
load_dotenv()
db = wrds.Connection(wrds_username=os.getenv('WRDS_USERNAME'))

Loading library list...
Done


# 單年測試

In [3]:
years = range(1996, 1997)
all_data = []

for year in years:
    print(f"正在抓取 {year} 年的資料...")
    start_time = time.time()

    sql_query = f"""
    WITH eom_dates AS (
        SELECT secid, date_trunc('month', date) AS month_start, MAX(date) AS eom_date
        FROM optionm.vsurfd{year}
        WHERE date >= '{year}-01-01' AND date <= '{year}-12-31'
        GROUP BY secid, date_trunc('month', date)
    ),
    filtered_ivs AS (
        SELECT v.secid, v.date, v.days, v.delta, v.impl_volatility
        FROM optionm.vsurfd{year} v
        INNER JOIN eom_dates e
            ON v.secid = e.secid AND v.date = e.eom_date
        WHERE v.days != 10
        AND ABS(v.delta) IN (10, 15, 20, 25, 30, 35, 40, 45, 50)
    ),
    raw_link AS (
        SELECT secid, permno, sdate, edate,
            LAG(edate) OVER (
                PARTITION BY permno
                ORDER BY sdate ASC, secid ASC -- 加上 secid ASC 消除同日期的隨機性
            ) AS prev_edate
        FROM wrdsapps.opcrsphist
        WHERE score = 1
    ),
    valid_link AS (
        SELECT secid, permno, sdate, edate
        FROM raw_link
        WHERE prev_edate IS NULL OR sdate > prev_edate
    ),
    target_permnos AS (
        SELECT DISTINCT vl.permno
        FROM valid_link AS vl
        INNER JOIN crsp.msenames n
            ON vl.permno = n.permno
            -- 確保抓的是這家公司在該年度的正確屬性 (因為公司可能轉板)
            AND n.namedt <= '{year}-12-31'
            AND n.nameendt >= '{year}-01-01'
        WHERE vl.sdate <= '{year}-12-31'
        AND (vl.edate >= '{year}-01-01' OR vl.edate IS NULL)
        -- 限制為美國三大交易所普通股
        AND n.shrcd IN (10, 11) -- Ensure only common stocks
        AND n.exchcd IN (1, 2, 3) -- Ensure only NYSE, AMEX, and NASDAQ stocks
        AND n.siccd NOT IN (6770, 6799, 6722, 6726) -- Exclude financial stocks (SIC codes for banks and financial institutions)
    ),
    daily_returns AS (
        SELECT d.permno,
            date_trunc('month', d.date) AS month_start,
            (1 + COALESCE(d.ret, 0)) * (1 + COALESCE(dl.dlret, 0)) AS gross_ret
        FROM crsp.dsf d
        -- INNER JOIN target_permnos tp ON d.permno = tp.permno -- filter early for performance
        LEFT JOIN crsp.dsedelist dl
            ON d.permno = dl.permno AND d.date = dl.dlstdt
        WHERE d.date >= '{year}-01-01' AND d.date <= '{year}-12-31'
    ),
    monthly_accumulated AS (
        SELECT permno,
            date_trunc('month', month_start)::date AS crsp_date,
            EXP(SUM(LN(GREATEST(gross_ret, 1e-10)))) - 1 AS crsp_monthly_return
        FROM daily_returns
        GROUP BY permno, month_start
    ),
    -- 計算所有股票的月度市值與交易所資訊
    crsp_me AS (
        SELECT
            m.permno,
            date_trunc('month', m.date) AS month_start,
            ABS(m.prc) * m.shrout AS me,
            n.exchcd,
            n.shrcd
        FROM crsp.msf m
        -- INNER JOIN target_permnos tp ON m.permno = tp.permno -- filter early for performance
        LEFT JOIN crsp.msenames n
            ON m.permno = n.permno
            AND m.date >= n.namedt AND m.date <= n.nameendt
        WHERE m.date >= '{year}-01-01' AND m.date <= '{year}-12-31'
        AND m.prc IS NOT NULL AND m.shrout IS NOT NULL
        AND n.shrcd IN (10, 11)
        AND n.exchcd IN (1, 2, 3)
        AND n.siccd NOT IN (6770, 6799, 6722, 6726)
    ),
    -- 算出 NYSE 普通股的 20, 50, 80 分位數
    nyse_bkp AS (
        SELECT
            month_start,
            PERCENTILE_CONT(0.20) WITHIN GROUP (ORDER BY me) AS p20,
            PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY me) AS p50,
            PERCENTILE_CONT(0.80) WITHIN GROUP (ORDER BY me) AS p80
        FROM crsp_me
        WHERE exchcd = 1 AND shrcd IN (10, 11)
        GROUP BY month_start
    )

    -- 最終整合：加上規模分組標籤
    SELECT
        f.secid,
        l.permno,
        f.date AS opt_date,
        f.days,
        f.delta,
        f.impl_volatility,
        m.crsp_date,
        m.crsp_monthly_return,
        c_me.me AS market_cap,
        c_me.shrcd,

        -- 依據市值落點給予分類標籤
        CASE
            WHEN c_me.me > bkp.p80 THEN 'Mega'
            WHEN c_me.me > bkp.p50 AND c_me.me <= bkp.p80 THEN 'Large'
            WHEN c_me.me > bkp.p20 AND c_me.me <= bkp.p50 THEN 'Small'
            WHEN c_me.me <= bkp.p20 THEN 'Micro'
            ELSE NULL
        END AS size_group,

        -- 判斷是否為非微型股 (Non-micro)
        CASE
            WHEN c_me.me > bkp.p20 THEN TRUE
            ELSE FALSE
        END AS is_non_micro

    FROM filtered_ivs f
    INNER JOIN valid_link l
        ON f.secid = l.secid
        AND f.date >= l.sdate
        AND (f.date <= l.edate OR l.edate IS NULL)
    INNER JOIN target_permnos tp -- 提早過濾掉非目標的 Permno
        ON l.permno = tp.permno
    LEFT JOIN monthly_accumulated m
        ON l.permno = m.permno
        AND date_trunc('month', f.date) = m.crsp_date
    -- 對接個別市值
    INNER JOIN crsp_me c_me
        ON l.permno = c_me.permno
        AND date_trunc('month', f.date) = c_me.month_start
    -- 對接全市場的 NYSE 分界點
    LEFT JOIN nyse_bkp bkp
        ON date_trunc('month', f.date) = bkp.month_start
    """

    try:
        print("正在執行 SQL 查詢與雲端運算...")
        query_start = time.time()
        df = db.raw_sql(sql_query, date_cols=['opt_date', 'crsp_date'])
        query_time = time.time() - query_start
        print(f"SQL 查詢完成 耗時: {query_time:.2f} 秒。共抓取 {len(df)} 筆資料。")

        if not df.empty:
            print("🗜️ 正在進行資料型別壓縮...")

            # 使用 Pandas 的 Nullable Integer 格式 (首字母大寫的 Int)，避免 NaN 導致轉型失敗
            df['secid'] = df['secid'].astype('Int32')
            df['permno'] = df['permno'].astype('Int32')
            df['days'] = df['days'].astype('Int16')
            df['market_cap'] = df['market_cap'].astype('float32')  # 市值通常很大，使用 float32 以節省空間
            df['delta'] = df['delta'].round().astype('Int16')  # 使用 Int16 確保安全
            df['size_group'] = df['size_group'].astype('category')
            df['is_non_micro'] = df['is_non_micro'].astype('bool')
            df['shrcd'] = df['shrcd'].astype('Int16')  # 交易所代碼通常不大，使用 Int16 足夠

            # 浮點數轉換為 float32
            df['impl_volatility'] = df['impl_volatility'].astype('float32')
            df['crsp_monthly_return'] = df['crsp_monthly_return'].astype('float32')

            # 3. 儲存檔案
            # file_name = os.path.join(OptionPath.IVS, f'option_ivs_crsp_{year}.parquet')
            file_name = f'option_ivs_crsp_{year}.parquet'
            df.to_parquet(file_name, engine='pyarrow')

            # 4. 結算時間與檔案大小
            total_time = time.time() - start_time
            file_size_mb = os.path.getsize(file_name) / (1024 * 1024)
            ram_mb = df.memory_usage(deep=True).sum() / 1024**2

            print("-" * 40)
            print("總結：")
            print(f"▶ 總耗時: {total_time:.2f} 秒")
            print(f"▶ 資料筆數: {len(df):,} 筆")
            print(f"▶ 檔案大小 ({file_name}): {file_size_mb:.2f} MB")
            print(f"▶ DataFrame 記憶體大小: {ram_mb:.2f} MB")
            print("▶ 壓縮後的資料型別 (dtypes):")
            print(df.dtypes)
            print("-" * 40)

        else:
            print("測試區間內沒有抓取到符合條件的資料。")

    except Exception as e:
        print(f"抓取資料時發生錯誤: {e}")

正在抓取 1996 年的資料...
正在執行 SQL 查詢與雲端運算...
SQL 查詢完成 耗時: 593.43 秒。共抓取 3700440 筆資料。
🗜️ 正在進行資料型別壓縮...
----------------------------------------
總結：
▶ 總耗時: 594.13 秒
▶ 資料筆數: 3,700,440 筆
▶ 檔案大小 (option_ivs_crsp_1996.parquet): 31.25 MB
▶ DataFrame 記憶體大小: 201.15 MB
▶ 壓縮後的資料型別 (dtypes):
secid                           Int32
permno                          Int32
opt_date               datetime64[ns]
days                            Int16
delta                           Int16
impl_volatility               float32
crsp_date              datetime64[ns]
crsp_monthly_return           float32
market_cap                    float32
shrcd                           Int16
size_group                   category
is_non_micro                     bool
dtype: object
----------------------------------------


# SQL 拆解

## eom_dates

In [ ]:
# -- 步驟 1: 找出每檔標的 (secid) 每個月的最後一個交易日
year = 2020

sql_eom = f"""
SELECT secid, date_trunc('month', date) AS month_start, MAX(date) AS eom_date
FROM optionm.vsurfd{year}
WHERE date >= '{year}-01-01' AND date <= '{year}-12-31'
GROUP BY secid, date_trunc('month', date)
"""
df_eom = db.raw_sql(sql_eom, date_cols=["month_start", "eom_date"])
df_eom.head()

/usr/local/lib/python3.12/dist-packages/wrds/sql.py:590: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  for chunk in df:


,secid,month_start,eom_date
0,112364.0,2020-04-01 00:00:00-04:00,2020-04-30
1,108254.0,2020-05-01 00:00:00-04:00,2020-05-29
2,207444.0,2020-02-01 00:00:00-05:00,2020-02-28
3,213653.0,2020-12-01 00:00:00-05:00,2020-12-31
4,107544.0,2020-01-01 00:00:00-05:00,2020-01-31


## filtered_ivs

In [35]:
# -- 步驟 2: 依照條件篩選 IVS 表面資料

sql_eom = f"""
WITH eom_dates AS (
    -- 步驟 1: 找出每檔標的 (secid) 每個月的最後一個交易日
    SELECT secid, date_trunc('month', date) AS month_start, MAX(date) AS eom_date
    FROM optionm.vsurfd{year}
    WHERE date >= '{year}-01-01' AND date <= '{year}-12-31'
    GROUP BY secid, date_trunc('month', date)
)
SELECT v.secid, v.date, v.days, v.delta, v.impl_volatility
FROM optionm.vsurfd{year} v
INNER JOIN eom_dates e
    ON v.secid = e.secid AND v.date = e.eom_date
WHERE v.days != 10
    AND ABS(v.delta) IN (10, 15, 20, 25, 30, 35, 40, 45, 50)
"""
filter_ivs = db.raw_sql(sql_eom, date_cols=["month_start", "eom_date"])
filter_ivs.head()

,secid,date,days,delta,impl_volatility
0,5139.0,2020-01-31,30.0,-50.0,0.306146
1,5139.0,2020-01-31,30.0,-45.0,0.305792
2,5139.0,2020-01-31,30.0,-40.0,0.30545
3,5139.0,2020-01-31,30.0,-35.0,0.307475
4,5139.0,2020-01-31,30.0,-30.0,0.474083


## raw_link to CRSP

In [63]:
# -- 步驟 3: 呼叫 WRDS 標準的 OptionMetrics-CRSP 連結表 opcrsphist
sql_eom = f"""
SELECT secid, permno, sdate, edate,
        LAG(edate) OVER (
            PARTITION BY permno
            ORDER BY sdate ASC, secid ASC -- 加上 secid ASC 消除同日期的隨機性
        ) AS prev_edate
FROM wrdsapps.opcrsphist
WHERE score = 1
"""
raw_link = db.raw_sql(sql_eom, date_cols=["month_start", "eom_date"])
raw_link.head()

,secid,permno,sdate,edate,prev_edate
0,104332.0,10001,1996-01-02,2017-08-03,<NA>
1,110326.0,10002,1996-01-02,2013-02-15,<NA>
2,6774.0,10009,1996-01-01,2000-11-03,<NA>
3,6518.0,10011,1996-01-01,1998-02-06,<NA>
4,103782.0,10012,1996-01-02,2005-08-02,<NA>


In [64]:
print(raw_link[raw_link['permno'] == 51596])

          secid  permno       sdate       edate  prev_edate
14066    5519.0   51596  1996-01-01  1999-10-07        <NA>
14067    5519.0   51596  1999-10-08  2007-05-30  1999-10-07
14068  106695.0   51596  1999-10-08  2007-05-30  2007-05-30


In [65]:
subset_duplicates = raw_link[raw_link.duplicated(subset=['permno', 'sdate', 'edate'], keep=False)]
subset_duplicates.sort_values(by=['secid', 'permno', 'sdate'])

,secid,permno,sdate,edate,prev_edate
14067,5519.0,51596,1999-10-08,2007-05-30,1999-10-07
15014,5846.0,70675,1996-01-01,1998-11-16,<NA>
14068,106695.0,51596,1999-10-08,2007-05-30,2007-05-30
15015,110171.0,70675,1996-01-01,1998-11-16,1998-11-16


In [70]:
# -- 步驟 4: 移除重複的月份資料 (同一個 permno 在同一個月可能有多筆紀錄，保留 edate 最晚的那筆)
sql_eom = f"""
WITH raw_link AS (
    SELECT secid, permno, sdate, edate,
        LAG(edate) OVER (PARTITION BY permno ORDER BY sdate) AS prev_edate
    FROM wrdsapps.opcrsphist
    WHERE score = 1
)
SELECT secid, permno, sdate, edate
FROM raw_link
WHERE prev_edate IS NULL OR sdate > prev_edate
"""
valid_link = db.raw_sql(sql_eom, date_cols=["month_start", "eom_date"])
valid_link.head()

,secid,permno,sdate,edate
0,104332.0,10001,1996-01-02,2017-08-03
1,110326.0,10002,1996-01-02,2013-02-15
2,6774.0,10009,1996-01-01,2000-11-03
3,6518.0,10011,1996-01-01,1998-02-06
4,103782.0,10012,1996-01-02,2005-08-02


## 篩選需要計算報酬的公司

In [ ]:
# -- 步驟 5: 只列出該年度需要計算日報酬的 PERMNO 清單
year = 2020
sql_eom = f"""
WITH raw_link AS (
    SELECT secid, permno, sdate, edate,
           LAG(edate) OVER (PARTITION BY permno ORDER BY sdate) AS prev_edate
    FROM wrdsapps.opcrsphist
    WHERE score = 1
),
valid_link AS (
    SELECT secid, permno, sdate, edate
    FROM raw_link
    WHERE prev_edate IS NULL OR sdate > prev_edate
)
SELECT DISTINCT vl.permno
FROM valid_link AS vl
-- 在名單階段就直接對接名稱表，剔除 ETF 與 ADR
INNER JOIN crsp.msenames AS n
    ON vl.permno = n.permno
    -- 確保我們抓的是這家公司在該年度的正確屬性 (因為公司可能轉板)
    AND n.namedt <= '{year}-12-31'
    AND n.nameendt >= '{year}-01-01'
WHERE vl.sdate <= '{year}-12-31'
    AND (vl.edate >= '{year}-01-01' OR vl.edate IS NULL)
    -- 限制為美國三大交易所普通股
    AND n.shrcd IN (10, 11)
    AND n.exchcd IN (1, 2, 3)
"""
target_permnos = db.raw_sql(sql_eom, date_cols=["month_start", "eom_date"])
target_permnos


,permno
0,10026
1,10028
2,10032
3,10044
4,10051
...,...
3910,93422
3911,93423
3912,93426
3913,93434


## 計算月報酬

In [ ]:
# -- 步驟 6: 每個 PERMNO 每日的 CRSP 報酬 (包含除名報酬，遺漏補 0)，並使用 EXP(SUM(LN())) 連乘後減 1 得到月報酬
year = 2020
end_date = '2020-02-28' # example
start_date = '2020-01-01'

sql_eom = f"""
WITH raw_link AS (
    SELECT secid, permno, sdate, edate,
           LAG(edate) OVER (PARTITION BY permno ORDER BY sdate) AS prev_edate
    FROM wrdsapps.opcrsphist
    WHERE score = 1
),
valid_link AS (
    SELECT secid, permno, sdate, edate
    FROM raw_link
    WHERE prev_edate IS NULL OR sdate > prev_edate
),
target_permnos AS (
    SELECT DISTINCT permno
    FROM valid_link
    WHERE sdate <= '{end_date}' AND (edate >= '{start_date}' OR edate IS NULL)
),
daily_returns AS (
    -- 步驟 4: 計算每日總毛報酬 (包含除名報酬，遺漏補 0)
    SELECT
        d.permno,
        date_trunc('month', d.date) AS month_start,
        -- 公式: (1 + 每日報酬) * (1 + 除名報酬)
        (1 + COALESCE(d.ret, 0)) * (1 + COALESCE(dl.dlret, 0)) AS gross_ret
    FROM crsp.dsf d
    INNER JOIN target_permnos tp ON d.permno = tp.permno
    LEFT JOIN crsp.dsedelist dl
        ON d.permno = dl.permno AND d.date = dl.dlstdt
    WHERE d.date >= '{year}-01-01' AND d.date <= '{year}-12-31'
)
SELECT
    permno,
    date_trunc('month', month_start)::date AS crsp_date,,
    -- 使用 EXP(SUM(LN())) 計算連乘
    EXP(SUM(LN(GREATEST(gross_ret, 1e-10)))) - 1 AS crsp_monthly_return
FROM daily_returns
GROUP BY permno, month_start
"""
monthly_return_df = db.raw_sql(sql_eom, date_cols=["month_start", "eom_date"])
monthly_return_df


/usr/local/lib/python3.12/dist-packages/wrds/sql.py:590: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  for chunk in df:


,permno,month_start,crsp_monthly_return
0,14282,2020-08-01 00:00:00-04:00,0.002251
1,18182,2020-01-01 00:00:00-05:00,0.001729
2,15677,2020-04-01 00:00:00-04:00,0.076425
3,19122,2020-07-01 00:00:00-04:00,0.015506
4,14332,2020-06-01 00:00:00-04:00,-0.018588
...,...,...,...
88986,88683,2020-06-01 00:00:00-04:00,-0.000003
88987,17150,2020-06-01 00:00:00-04:00,0.305522
88988,14319,2020-04-01 00:00:00-04:00,0.345571
88989,78955,2020-04-01 00:00:00-04:00,0.043092


# 檢查特定 permno 是否連續

In [ ]:
year = 2024
sql = f"""
SELECT permno, namedt, nameendt, shrcd, exchcd, ticker, comnam, siccd
FROM crsp.msenames
WHERE permno IN (
    SELECT DISTINCT permno
    FROM crsp.msf
    WHERE date BETWEEN '{year}-01-01' AND '{year}-12-31'
)
ORDER BY permno, namedt
"""
msenames_df = db.raw_sql(sql, date_cols=["namedt", "nameendt"])
msenames_df[msenames_df['permno'] == 18370]

,permno,namedt,nameendt,shrcd,exchcd,ticker,comnam,siccd
8981,18370,2019-01-09,2019-12-09,11,3,SAMA,SCHULTZE SPEC PUR ACQ CORP,9999
8982,18370,2019-12-10,2020-07-30,11,3,SAMA,SCHULTZE SPEC PUR ACQ CORP,9999
8983,18370,2020-07-31,2020-12-17,11,3,SAMA,SCHULTZE SPEC PUR ACQ CORP,9999
8984,18370,2020-12-18,2023-08-24,12,3,CLVR,CLEVER LEAVES HOLDINGS INC,9999
8985,18370,2023-08-25,2024-05-16,12,3,CLVR,CLEVER LEAVES HOLDINGS INC,9999


In [17]:
msenames_df[(msenames_df['siccd'] > 6000) & (msenames_df['siccd'] < 7000)]

,permno,namedt,nameendt,shrcd,exchcd,ticker,comnam,siccd
36,10065,1930-01-02,1962-07-01,14,1,<NA>,ADAMS EXPRESS CO,6720
37,10065,1962-07-02,1968-01-01,14,1,ADX,ADAMS EXPRESS CO,6723
38,10065,1968-01-02,2001-08-23,14,1,ADX,ADAMS EXPRESS CO,6723
39,10065,2001-08-24,2002-01-01,14,1,ADX,ADAMS EXPRESS CO,6726
40,10065,2002-01-02,2004-06-09,14,1,ADX,ADAMS EXPRESS CO,6726
...,...,...,...,...,...,...,...,...
37494,93425,2024-06-17,2024-12-31,74,4,BNO,UNITED STATES BRENT OIL FUND LP,6221
37504,93429,2018-09-17,2021-03-28,11,5,CBOE,C B O E GLOBAL MARKETS INC,6231
37505,93429,2021-03-29,2023-03-26,11,5,CBOE,C B O E GLOBAL MARKETS INC,6211
37506,93429,2023-03-27,2023-08-10,11,5,CBOE,C B O E GLOBAL MARKETS INC,6211
